# A10: Data Understanding on the unified dataset (Tanner Edwards)

Personal notebook. Re-runs Matt's Data Understanding (`Sorensen-A10.ipynb` §2–§3, plan steps 8–9) on the **unified dataset** built by `00_build_dataset.ipynb`, which adds four sources and grows the usable population from 5,766 to about 6,800 films. Same helpers, same review flags, same privacy rule.

**Run `00_build_dataset.ipynb` first.** This notebook only reads `data/clean/films_unified.csv` and the raw files; it writes nothing.

| Section | What | Matt's equivalent |
|---|---|---|
| 1 | Setup and Matt's profile helpers | §1, §3 helpers |
| 2 | Data Receipt for the **new** sources | §2 |
| 3a | Field inventory: all columns, role at the financing decision | §3a |
| 3b | The label on the expanded population | §3b |
| 3c–3f | Numeric, categorical, identifier/text and date fields | §3c–§3h |
| 3g | **New:** do the sources agree with each other? | — |
| 3h | **New:** who has which data (coverage by period and budget tier) | §4a idea |
| 3i | Flag register | §3i |
| 3j–3k | Findings and proposed dispositions (need Tanner + Matt sign-off) | §3j–§3k |

This section **observes and records; it doesn't clean.**

## 1. Setup

In [ ]:
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

RAW = Path("../data/raw")
CLEAN = Path("../data/clean")
YEARS = (2000, 2023)

films = pd.read_csv(CLEAN / "films_unified.csv", low_memory=False)
for c in ["release_dt", "tn_Release Date", "bom_first_date", "sl_original_release_date",
          "rt_releaseDateTheaters", "rt_releaseDateStreaming", "rtrev_first_review", "slrev_first_review"]:
    films[c] = pd.to_datetime(films[c], errors="coerce")
print(f"films_unified.csv: {len(films):,} films x {films.shape[1]} columns; {films['usable'].sum():,} usable")

Matt's chart style and profile helpers, copied unchanged from `Sorensen-A10.ipynb` §3 so the two profiles are directly comparable.

In [ ]:
SERIES = {"train 2000-15": "#2a78d6", "test 2016-19": "#eb6834", "stress 2020-23": "#1baf7a"}
BLUE, BLUE_LIGHT, INK, INK2 = "#2a78d6", "#cde2fb", "#0b0b0b", "#52514e"
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#b5b4ad", "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.6,
    "axes.titlesize": 10, "axes.titlelocation": "left", "axes.labelsize": 9,
    "xtick.labelsize": 8, "ytick.labelsize": 8,
    "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
})

SKEW_FLAG, RARE_FLAG = 0.5, 0.05   # Ch 6 review flags
FLAGS = []

def flag(file, field, issue, evidence):
    FLAGS.append({"file": file, "field": field, "issue": issue, "evidence": evidence})


def numeric_profile(df, cols, zero_as_missing=()):
    rows = []
    for c in cols:
        s = df[c].astype(float)
        if c in zero_as_missing:
            s = s.mask(s == 0)
        x = s.dropna()
        q1, q3 = x.quantile([0.25, 0.75])
        iqr = q3 - q1
        rows.append({
            "field": c, "n": len(x), "% missing": round(100 * s.isna().mean(), 1),
            "mean": x.mean(), "median": x.median(), "std": x.std(),
            "min": x.min(), "Q1": q1, "Q3": q3, "max": x.max(), "IQR": iqr, "skew": x.skew(),
            "# above Q3+1.5 IQR": int((x > q3 + 1.5 * iqr).sum()),
            "# below Q1-1.5 IQR": int((x < q1 - 1.5 * iqr).sum()),
        })
    out = pd.DataFrame(rows).set_index("field")
    out["skew flag"] = out["skew"].abs() > SKEW_FLAG
    return out


def category_profile(s, name, multi=False, names_hidden=False):
    n = s.notna().sum()
    vals = s.dropna().astype(str)
    if multi:
        vals = vals.str.split(",").explode().str.strip()
        vals = vals[vals.ne("")]
    vc = vals.value_counts()
    share = vc / n
    row = {
        "field": name, "% missing": round(100 * s.isna().mean(), 1), "multi-label": multi,
        "levels": len(vc),
        "top level": "(person, withheld)" if names_hidden else (vc.index[0] if len(vc) else None),
        "top level % of rows": round(100 * share.iloc[0], 1) if len(vc) else None,
        "levels < 5% of rows": int((share < RARE_FLAG).sum()),
        "% of rows in a level < 5%": None if multi else round(100 * share[share < RARE_FLAG].sum(), 1),
    }
    return row, vc


def id_text_profile(df, cols):
    rows = []
    for c in cols:
        s = df[c]
        nn = s.dropna().astype(str)
        rows.append({
            "field": c, "% missing": round(100 * s.isna().mean(), 1),
            "unique values": nn.nunique(),
            "rows sharing a value": int(nn.duplicated(keep=False).sum()),
            "median length (chars)": nn.str.len().median(),
        })
    return pd.DataFrame(rows).set_index("field")


def box_hist(ax_box, ax_hist, x, title, xlabel, ylabel, log=False):
    x = x.dropna().astype(float)
    if log and (x <= 0).any():
        title += f"\n({(x <= 0).sum():,} values <= 0 not shown on the log scale)"
        x = x[x > 0]
    bins = int(np.ceil(len(x) ** (1 / 3)))
    edges = np.logspace(np.log10(x.min()), np.log10(x.max()), bins + 1) if log else bins
    ax_hist.hist(x, bins=edges, color=BLUE, edgecolor="white", linewidth=0.4)
    ax_box.boxplot(x, orientation="horizontal", widths=0.6, patch_artist=True,
                   boxprops=dict(facecolor=BLUE_LIGHT, edgecolor=BLUE),
                   medianprops=dict(color=INK, linewidth=1.5),
                   whiskerprops=dict(color=BLUE), capprops=dict(color=BLUE),
                   flierprops=dict(marker="o", markersize=2.5, markerfacecolor=BLUE,
                                   markeredgecolor="none", alpha=0.35))
    ax_hist.axvline(x.median(), color=INK, lw=1, label=f"median {x.median():,.3g}")
    ax_hist.axvline(x.mean(), color=INK, lw=1, ls="--", label=f"mean {x.mean():,.3g}")
    if log:
        ax_hist.set_xscale("log")
    ax_box.set_yticks([])
    ax_box.grid(False)
    ax_box.set_title(title)
    ax_hist.set_xlabel(xlabel)
    ax_hist.set_ylabel(ylabel)
    ax_hist.legend(fontsize=7, frameon=False)


def profile_grid(specs, ylabel, ncols=3):
    nrows = int(np.ceil(len(specs) / ncols))
    fig = plt.figure(figsize=(4.3 * ncols, 3.1 * nrows), layout="constrained")
    subfigs = np.atleast_1d(fig.subfigures(nrows, ncols)).ravel()
    for sf, (x, title, xlabel, log) in zip(subfigs, specs):
        ax_box, ax_hist = sf.subplots(2, 1, sharex=True, height_ratios=[1, 4])
        box_hist(ax_box, ax_hist, x, title, xlabel, ylabel, log)
    for sf in subfigs[len(specs):]:
        sf.set_visible(False)
    plt.show()


def count_plot(ax, vc, n_rows, title, top=10, xlabel="% of rows"):
    shares = (100 * vc / n_rows).head(top)[::-1]
    ax.barh(shares.index.astype(str).str.slice(0, 35), shares.values, color=BLUE, height=0.7)
    ax.axvline(100 * RARE_FLAG, color=INK2, ls=":", lw=1)
    for y, v in enumerate(shares.values):
        ax.text(v, y, f" {v:.1f}%", va="center", fontsize=7, color=INK2)
    ax.set_title(title + (f"  (top {top} of {len(vc):,})" if len(vc) > top else ""))
    ax.set_xlabel(xlabel)
    ax.grid(axis="y", visible=False)

print("Profile helpers defined.")

## 2. Data Receipt for the new sources (plan step 8)

Matt's §2 covers TMDB, RT movies (andrezaza) and RT reviews. These are the sources added since. Licenses were read from the Kaggle API (`licenseName`) and the GitHub API on 2026-11-23.

| Field | The Numbers | Box Office Mojo | Wikidata snapshot | RT movies (stefanoleone992) | RT critic reviews (stefanoleone992) |
|---|---|---|---|---|---|
| **File** | `the_numbers/Top Movies (Cleaned Data).csv` | `boxofficemojo/revenues_per_day.csv.gz` | `wikidata/wikidata_film_financials_2026-11-21.csv` | `stefanoleone992/rotten_tomatoes_movies.csv` | `stefanoleone992/rotten_tomatoes_critic_reviews.csv` |
| **Publisher** | michaelmatta0 (Kaggle) | tjwaterman99 (GitHub release) | Our own SPARQL query (`tools/wikidata_films.rq`) | stefanoleone992 (Kaggle) | same |
| **Original source** | the-numbers.com (Nash Information Services) | boxofficemojo.com (IMDb) | Wikidata (Wikimedia, crowd-edited) | Rotten Tomatoes | Rotten Tomatoes |
| **URL** | kaggle.com/datasets/michaelmatta0/movies-ultimate-metrics-features-and-metadata | github.com/tjwaterman99/boxofficemojo-scraper | query.wikidata.org | kaggle.com/datasets/stefanoleone992/rotten-tomatoes-movies-and-critic-reviews-dataset | same |
| **Access date** | 2026-11-21 | 2026-11-20 | 2026-11-21 (snapshot committed to git) | 2026-11-20 | 2026-11-20 |
| **License (as stated)** | MIT | **None stated** | CC0 1.0 | CC0 1.0 | CC0 1.0 |
| **Claimed grain** | One row per film | One row per film **per day** | One row per film with a USD budget and a TMDB id | One row per RT film (link) | One row per critic review |
| **Population** | ~6.6k "top" films by budget, all years | Films on the US daily chart, 2000–2025 | Wikidata films 2000–2023 with a USD cost | ~17.7k RT films (2020 scrape) | Reviews of those films |
| **Sensitive fields** | None | None | None | `directors`, `authors`, `actors`: public professionals | `critic_name`, `review_content`: named people and copyrighted text |
| **Rights caveats** | Scraped; The Numbers' terms aren't overridden by the uploader's MIT | Scraped; no license, so academic fair use only | — | Scores belong to RT | Review text is copyrighted; not copied into the unified file |
| **Permitted use here** | Academic, non-commercial. Cite the-numbers.com. | Same, and cite Box Office Mojo | Cite Wikidata | Same as Matt's RT files | Same; counts only |

**Also note:** `get_data.py` downloads `rotten_tomatoes_movie_reviews.csv` from **priyamchoksi** (license MIT), while Matt's receipt cites the **andrezaza** upload (CC0). Same row count (1,444,963). The receipt should cite one; open question for Matt.

### 2a. File fingerprints
SHA-256 for every raw file, Matt's originals included, so all of them can be checked as unchanged at submission.

In [ ]:
def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(chunk):
            h.update(block)
    return h.hexdigest()

FILES = sorted(p for p in RAW.rglob("*") if p.is_file() and p.name != ".gitkeep")
fingerprints = pd.DataFrame([{"file": str(p.relative_to(RAW)), "MB": round(p.stat().st_size / 1e6, 1),
                              "sha256": sha256(p)} for p in FILES])
with pd.option_context("display.max_colwidth", 70):
    display(fingerprints)

### 2b. Grain check (new sources)
Does one row equal what the source claims? Box Office Mojo's `id` turned out to be per row, not per film (it was collapsed to one row per title in the build).

In [ ]:
tn_raw = pd.read_csv(RAW / "the_numbers/Top Movies (Cleaned Data).csv", low_memory=False)
bom_raw = pd.read_csv(RAW / "boxofficemojo/revenues_per_day.csv.gz")
wd_raw = pd.read_csv(RAW / "wikidata/wikidata_film_financials_2026-11-21.csv")
sl_raw = pd.read_csv(RAW / "stefanoleone992/rotten_tomatoes_movies.csv", low_memory=False)
slr_raw = pd.read_csv(RAW / "stefanoleone992/rotten_tomatoes_critic_reviews.csv", low_memory=False,
                      usecols=["rotten_tomatoes_link", "critic_name", "review_date", "review_type", "top_critic"])

def grain_check(df, key):
    return {"rows": len(df), "key": key, "unique keys": df[key].nunique(),
            "rows with a repeated key": int(df[key].duplicated(keep=False).sum()),
            "exact duplicate rows": int(df.duplicated().sum())}

grain = pd.DataFrame({
    "The Numbers (Movie URL)": grain_check(tn_raw, "Movie URL"),
    "Box Office Mojo (id)": grain_check(bom_raw, "id"),
    "Box Office Mojo (title + date)": grain_check(bom_raw.assign(k=bom_raw["title"] + "|" + bom_raw["date"]), "k"),
    "Wikidata (tmdb)": grain_check(wd_raw, "tmdb"),
    "RT sl movies (link)": grain_check(sl_raw, "rotten_tomatoes_link"),
    "RT sl reviews (link + critic + date)": grain_check(
        slr_raw.assign(k=slr_raw["rotten_tomatoes_link"] + "|" + slr_raw["critic_name"].fillna("?") + "|" + slr_raw["review_date"].astype(str)), "k"),
})
grain

### 2c. Date coverage and placeholder dates (new sources)

In [ ]:
dates = {
    "The Numbers Release Date": pd.to_datetime(tn_raw["Release Date"], errors="coerce"),
    "Box Office Mojo date": pd.to_datetime(bom_raw["date"], errors="coerce"),
    "RT sl original_release_date": pd.to_datetime(sl_raw["original_release_date"], errors="coerce"),
    "RT sl review_date": pd.to_datetime(slr_raw["review_date"], errors="coerce"),
}
ACCESS = pd.Timestamp("2026-11-21")
pd.DataFrame({n: {"min": d.min().date(), "max": d.max().date(), "% missing": round(100 * d.isna().mean(), 1),
                  "before 1900": int((d.dt.year < 1900).sum()), "after 2023": int((d.dt.year > 2023).sum()),
                  "after access date": int((d > ACCESS).sum())} for n, d in dates.items()}).T

### 2d. Population: where the usable films come from

Same funnel as Matt's §2f, but on the unified table, with the source of each film's budget and revenue. The label period split is the contract's.

In [ ]:
f = films.copy()
f["period"] = pd.cut(f["year"], [1999, 2015, 2019, 2023], labels=list(SERIES))
steps = [
    ("2000-2023 film with any budget or revenue (unified table)", pd.Series(True, index=f.index)),
    ("budget >= $1M (any source)", f["fin_budget"].ge(1_000_000)),
    ("revenue > 0 (any source)", f["fin_revenue"].gt(0)),
]
mask = pd.Series(True, index=f.index)
funnel = []
for label, cond in steps:
    mask &= cond
    funnel.append({"filter (cumulative)": label, "films remaining": int(mask.sum())})
display(pd.DataFrame(funnel))
assert mask.sum() == f["usable"].sum()

u = f[f["usable"]].copy()
display(pd.crosstab(u["fin_budget_src"], u["fin_revenue_src"], margins=True).rename_axis(index="budget from", columns="revenue from"))
display(pd.crosstab(u["period"], u["in_tmdb"].map({True: "TMDB film", False: "The Numbers only"}), margins=True))

# S4 restated on the unified table: budget >= $1M but no revenue from any source
b1 = f["fin_budget"].ge(1_000_000)
s4 = pd.DataFrame({p: {"budget >= $1M": int((b1 & (f["period"] == p)).sum()),
                       "with revenue": int((b1 & f["fin_revenue"].gt(0) & (f["period"] == p)).sum())}
                   for p in SERIES}).T
s4["% missing revenue"] = (100 * (1 - s4["with revenue"] / s4["budget >= $1M"])).round(1)
display(s4)
flag("Unified", "fin_revenue", "missing (S4)",
     f"{100 * (1 - (b1 & f['fin_revenue'].gt(0)).sum() / b1.sum()):.1f}% of budget >= $1M films have no revenue "
     f"from any source (Matt's TMDB-only figure: 43.6%)")

## 3. Univariate profile of every field (plan step 9)

**Which rows:** the usable films (2000–2023, budget ≥ $1M, revenue > 0 from any source), the population the screen will score. The label is shown by period. **Review flags (Ch 6):** |skew| > 0.5; any category level held by < 5% of non-missing rows. **Privacy:** director, writer, actor and critic names are profiled by counts only.

In [ ]:
pop = u
pop["multiple"] = pop["fin_revenue"] / pop["fin_budget"]
pop["log10_multiple"] = np.log10(pop["multiple"])
for k in [1, 2, 2.5, 3]:
    pop[f"loser_{k}x"] = pop["multiple"] < k
pop["money_loser"] = pop["loser_2.5x"]
pop["release_month"] = pop["release_dt"].dt.month
pop["n_genres"] = pop["genres"].str.split(",").str.len()
print(f"Population: {len(pop):,} films")
print(pop["period"].value_counts().sort_index().to_string())

### 3a. Field inventory
Every column in the unified file, with its source and what the contract lets it do at the financing decision.
- **allowed input**: on the contract's input list (budget, genre, director/company track record, runtime, release month/quarter, language, country)
- **candidate input (needs amendment)**: known before production, but not on the contract's list (e.g. MPAA rating, source material, franchise, writer, cast). Using it needs a dated amendment.
- **prohibited (post-release)**, **outcome part**, **key / join**, **build metadata**, **not used**

In [ ]:
A, CAND, P, K, O, B, N = ("allowed input", "candidate input (needs amendment)", "prohibited (post-release)",
                         "key / join", "outcome part (post-release)", "build metadata", "not used")

def source_of(c):
    for pre, name in [("tn_", "The Numbers"), ("bom_", "Box Office Mojo"), ("sl_", "RT sl movies"),
                      ("wd_", "Wikidata"), ("rtrev_", "RT reviews (rollup)"), ("slrev_", "RT sl reviews (rollup)"),
                      ("rt_", "RT movies"), ("fin_", "Combined")]:
        if c.startswith(pre):
            return name
    return "Build" if c in ("film_uid", "usable", "in_tmdb", "merged_ids", "release_dt", "year") else "TMDB"

ROLE = {
    # TMDB
    "film_uid": K, "id": K, "title": K, "original_title": K, "imdb_id": K, "release_date": A, "release_dt": A, "year": A,
    "budget": A, "revenue": O, "runtime": A, "genres": A, "production_companies": A, "production_countries": A,
    "original_language": A, "spoken_languages": N, "keywords": P, "vote_average": P, "vote_count": P,
    "popularity": P, "status": B, "adult": B, "overview": N, "tagline": N, "homepage": N, "poster_path": N,
    "backdrop_path": N,
    # combined / build
    "fin_budget": A, "fin_revenue": O, "fin_budget_src": B, "fin_revenue_src": B, "usable": B, "in_tmdb": B,
    "merged_ids": B,
    # The Numbers
    "tn_Movie Name": K, "tn_Release Date": A, "tn_Production Budget (USD)": A, "tn_Running Time (minutes)": A,
    "tn_Genre": A, "tn_Production/Financing Companies": A, "tn_Production Countries": A, "tn_Languages": A,
    "tn_MPAA Rating": CAND, "tn_Source": CAND, "tn_Creative Type": CAND, "tn_Production Method": CAND,
    "tn_Franchise": CAND, "tn_Keywords": P, "tn_Movie URL": K, "tn_year_diff": B,
    "tn_Domestic Releases": P, "tn_International Releases": P, "tn_Theater counts": P, "tn_Video Release": P,
    "tn_Legs": P, "tn_Opening Weekend (USD)": P, "tn_Domestic Share Percentage": P,
    "tn_Est. Domestic DVD Sales (USD)": P, "tn_Est. Domestic Blu-ray Sales (USD)": P,
    "tn_Total Est. Domestic Video Sales (USD)": P,
    # Box Office Mojo
    "bom_title": K, "bom_year_diff": B, "bom_first_date": P, "bom_distributor": CAND,
    # RT sl movies
    "sl_rotten_tomatoes_link": K, "sl_movie_title": K, "sl_year_diff": B, "sl_genres": A, "sl_runtime": A,
    "sl_production_company": A, "sl_directors": A, "sl_authors": CAND, "sl_actors": CAND,
    "sl_content_rating": CAND, "sl_original_release_date": A, "sl_movie_info": N, "sl_critics_consensus": P,
    "sl_streaming_release_date": P,
    # RT movies (andrezaza)
    "rt_id": K, "rt_title": K, "rt_matched_on": B, "rt_rule": B, "rt_year_diff": B, "rt_runtime_diff": B,
    "rt_director": A, "rt_writer": CAND, "rt_rating": CAND, "rt_ratingContents": CAND, "rt_genre": A,
    "rt_originalLanguage": A, "rt_runtimeMinutes": A, "rt_releaseDateTheaters": A,
    "rt_releaseDateStreaming": P, "rt_distributor": CAND, "rt_soundMix": N, "rt_boxOffice": O,
    "rt_boxOffice_usd": O,
    # Wikidata
    "wd_item": K, "wd_title": K, "wd_year": A, "wd_wd_budget_usd": A, "wd_wd_box_office_usd": O,
}
def role_of(c):
    if c in ROLE:
        return ROLE[c]
    if c.startswith(("tn_Domestic", "tn_International", "tn_Worldwide", "tn_Infl", "bom_domestic", "bom_days", "bom_max")):
        return O if "Gross" in c or "Box Office" in c or "gross" in c else P
    if c.startswith(("sl_tomatometer", "sl_audience", "rt_audience", "rt_tomato", "rtrev_", "slrev_")):
        return P
    return None

inventory = pd.DataFrame({"field": films.columns})
inventory["source"] = inventory["field"].map(source_of)
inventory["dtype"] = inventory["field"].map(lambda c: str(films[c].dtype))
inventory["% filled (usable films)"] = inventory["field"].map(lambda c: round(100 * pop[c].notna().mean(), 1))
inventory["role at financing decision"] = inventory["field"].map(role_of)
assert inventory["role at financing decision"].notna().all(), inventory.loc[inventory["role at financing decision"].isna(), "field"].tolist()
with pd.option_context("display.max_rows", 200, "display.max_colwidth", 60):
    display(inventory)
pd.crosstab(inventory["source"], inventory["role at financing decision"], margins=True)

### 3b. The label on the expanded population
Money-loser = combined revenue < 2.5 × combined budget, with 1×, 2×, 3× sensitivity. Compared with Matt's TMDB-only figures (64.6% losers overall), and split by where the financials came from.

In [ ]:
def label_summary(g):
    return pd.Series({
        "films": len(g), "median multiple": g["multiple"].median(),
        "Q1": g["multiple"].quantile(0.25), "Q3": g["multiple"].quantile(0.75),
        "skew (log10 multiple)": g["log10_multiple"].skew(),
        **{f"% losers at {k}x": 100 * g[f"loser_{k}x"].mean() for k in [1, 2, 2.5, 3]},
    })

label_tbl = pd.DataFrame({p: label_summary(g) for p, g in pop.groupby("period", observed=True)}
                         | {"all 2000-23": label_summary(pop)}).T
display(label_tbl.round(2))

pop["fin_source"] = np.where(pop["fin_budget_src"].eq("tmdb") & pop["fin_revenue_src"].eq("tmdb"),
                             "TMDB only (Matt's population)", "uses The Numbers / Wikidata")
display(pd.DataFrame({k: label_summary(g) for k, g in pop.groupby("fin_source")}).T.round(2))

extremes = pd.Series({
    "multiple < 0.01": (pop["multiple"] < 0.01).sum(), "multiple < 0.1": (pop["multiple"] < 0.1).sum(),
    "multiple > 20": (pop["multiple"] > 20).sum(), "multiple > 100": (pop["multiple"] > 100).sum(),
}, name="films")
display(extremes.to_frame())
cols = ["title", "year", "fin_budget", "fin_revenue", "fin_budget_src", "fin_revenue_src", "multiple"]
print("Lowest 5 multiples:"); display(pop.nsmallest(5, "multiple")[cols])
print("Highest 5 multiples:"); display(pop.nlargest(5, "multiple")[cols])
loser_rate = label_tbl.loc["all 2000-23", "% losers at 2.5x"]
flag("Unified", "multiple", "extreme values",
     f"{extremes.iloc[0]} films below 0.01x and {extremes.iloc[3]} above 100x: check for unit errors")
flag("Unified", "money_loser", "majority class",
     f"{loser_rate:.1f}% losers at 2.5x: a 20% flag can catch at most {20 / loser_rate:.0%} of them")

edges = np.logspace(-3, 3, 37)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2), layout="constrained", sharey=True)
for ax, (p, g) in zip(axes, pop.groupby("period", observed=True)):
    x = g["multiple"].clip(edges[0], edges[-1])
    ax.hist(x, bins=edges, weights=np.full(len(x), 100 / len(x)), color=SERIES[p], edgecolor="white", linewidth=0.4)
    ax.axvline(2.5, color=INK, lw=1, ls="--")
    ax.set_xscale("log")
    ax.set_title(f"{p}: n = {len(g):,}, {100 * g['money_loser'].mean():.0f}% losers")
    ax.set_xlabel("revenue / budget (log scale)")
axes[0].set_ylabel("% of films in period")
fig.suptitle("Label on the unified population (dashed line = 2.5x cut-off)", x=0.01, ha="left", fontsize=11)
plt.show()

### 3c. Numeric fields
Money and count fields on log scales. Runtime 0 = unknown. Post-release fields are profiled for quality only.

In [ ]:
num_cols = [c for c in films.columns if pd.api.types.is_numeric_dtype(films[c])
            and c not in ("id",) and not c.endswith("_year_diff") and c not in ("rt_runtime_diff",)
            and films[c].dtype != bool]
zero_missing = ("runtime", "tn_Running Time (minutes)", "sl_runtime", "rt_runtimeMinutes", "budget", "revenue")
prof_num = numeric_profile(pop, num_cols + ["multiple", "log10_multiple", "n_genres"], zero_as_missing=zero_missing)
prof_num.insert(0, "role", prof_num.index.map(lambda c: role_of(c) or "derived"))
with pd.option_context("display.float_format", "{:,.2f}".format, "display.max_rows", 80):
    display(prof_num)

checks = pd.Series({
    "runtime (TMDB) = 0 or missing": int(pop["runtime"].fillna(0).eq(0).sum()),
    "runtime < 60 min": int(pop["runtime"].between(1, 59).sum()),
    "runtime > 240 min": int((pop["runtime"] > 240).sum()),
    "fin_budget > $400M": int((pop["fin_budget"] > 400e6).sum()),
    "fin_budget is a round $1M multiple": int((pop["fin_budget"] % 1_000_000 == 0).sum()),
    "fin_revenue < $1,000": int((pop["fin_revenue"] < 1_000).sum()),
    "fin_revenue < $100,000": int((pop["fin_revenue"] < 100_000).sum()),
    "scores outside 0-100 (RT / sl)": int(sum((~pop[c].dropna().between(0, 100)).sum() for c in
                                              ["rt_audienceScore", "rt_tomatoMeter", "sl_tomatometer_rating", "sl_audience_rating"])),
}, name="films")
display(checks.to_frame())
flag("Unified", "fin_revenue", "implausible range",
     f"{checks['fin_revenue < $1,000']} films under $1,000; {checks['fin_revenue < $100,000']} under $100,000")
flag("Unified", "fin_budget", "estimate, not audited",
     f"{checks['fin_budget is a round $1M multiple']:,} of {len(pop):,} budgets are round $1M figures")
flag("Unified", "runtime", "implausible range",
     f"{checks['runtime (TMDB) = 0 or missing']} zero/missing, {checks['runtime < 60 min']} under 60 min, "
     f"{checks['runtime > 240 min']} over 240 min")

profile_grid([
    (pop["fin_budget"], "fin_budget (combined)", "US$ (log scale)", True),
    (pop["fin_revenue"], "fin_revenue (combined; outcome part)", "US$ (log scale)", True),
    (pop["runtime"].mask(pop["runtime"].eq(0)), "runtime (TMDB; 0 excluded)", "minutes", False),
    (pop["rt_tomatoMeter"], "rt_tomatoMeter (post-release)", "score, 0-100", False),
    (pop["rt_audienceScore"], "rt_audienceScore (post-release)", "score, 0-100", False),
    (pop["rtrev_n_reviews"], "rtrev_n_reviews (post-release)", "reviews (log scale)", True),
], ylabel="films")

### 3d. Categorical fields
Multi-label fields are split on commas, so shares add to more than 100%. Person fields (director, writer, actors) show counts only.

In [ ]:
cat_specs = [
    # (column, multi-label, names hidden)
    ("original_language", False, False), ("genres", True, False), ("production_countries", True, False),
    ("production_companies", True, False), ("spoken_languages", True, False), ("keywords", True, False),
    ("tn_MPAA Rating", False, False), ("tn_Source", False, False), ("tn_Genre", False, False),
    ("tn_Creative Type", False, False), ("tn_Production Method", True, False), ("tn_Franchise", False, False),
    ("tn_Production/Financing Companies", True, False), ("tn_Production Countries", True, False),
    ("tn_Languages", True, False), ("tn_Keywords", True, False),
    ("bom_distributor", False, False),
    ("sl_content_rating", False, False), ("sl_genres", True, False), ("sl_production_company", False, False),
    ("sl_tomatometer_status", False, False), ("sl_audience_status", False, False),
    ("rt_rating", False, False), ("rt_genre", True, False), ("rt_originalLanguage", False, False),
    ("rt_distributor", False, False), ("rt_soundMix", True, False),
    ("rt_director", True, True), ("rt_writer", True, True), ("sl_directors", True, True),
    ("sl_authors", True, True), ("sl_actors", True, True),
    ("fin_budget_src", False, False), ("fin_revenue_src", False, False),
]
rows, counts = [], {}
for col, multi, hidden in cat_specs:
    row, vc = category_profile(pop[col], col, multi, names_hidden=hidden)
    row["role"] = role_of(col)
    rows.append(row)
    counts[col] = vc
prof_cat = pd.DataFrame(rows).set_index("field")
with pd.option_context("display.max_rows", 60, "display.max_colwidth", 40):
    display(prof_cat)

# Track-record sparsity (counts only): directors and companies with a single film in the population
for col, label in [("rt_director", "RT directors"), ("production_companies", "TMDB production companies")]:
    vc = counts[col]
    print(f"{label}: {len(vc):,} distinct; {(vc == 1).mean():.0%} appear in only one usable film")

# The Numbers 'Franchise' looks like several names run together (e.g. 'BatmanDark Knight Trilogy')
fr = pop["tn_Franchise"].dropna()
glued = fr.str.contains(r"[a-z][A-Z]").sum()
print(f"tn_Franchise: {len(fr):,} films; {glued:,} values look like names glued together (lowercase then uppercase)")
flag("The Numbers", "tn_Franchise", "parse error", f"{glued:,} of {len(fr):,} values look like several franchise names run together")

fig, axes = plt.subplots(2, 3, figsize=(15, 7), layout="constrained")
for ax, col in zip(axes.ravel(), ["genres", "tn_MPAA Rating", "tn_Source", "tn_Creative Type", "original_language", "fin_budget_src"]):
    count_plot(ax, counts[col], pop[col].notna().sum(), col, xlabel="% of films with the level")
fig.suptitle("Selected categories, usable films (dotted line = 5% rare-level flag)", x=0.01, ha="left", fontsize=11)
plt.show()

### 3e. Identifier and free-text fields
Uniqueness, duplicates and length only. After the build's one-row-per-film step, no key should repeat.

In [ ]:
id_cols = ["id", "imdb_id", "title", "original_title", "tn_Movie URL", "rt_id", "sl_rotten_tomatoes_link",
           "wd_item", "overview", "tagline", "sl_movie_info", "sl_critics_consensus", "merged_ids"]
prof_id = id_text_profile(pop, id_cols)
display(prof_id)
for key in ["id", "imdb_id", "tn_Movie URL", "rt_id", "sl_rotten_tomatoes_link", "wd_item"]:
    shared = prof_id.loc[key, "rows sharing a value"]
    if shared:
        flag("Unified", key, "duplicate key", f"{shared} usable films share a {key}")
same_ty = pop.duplicated(["title", "year"], keep=False)
print(f"usable films sharing title and year with another film: {same_ty.sum()}")
if same_ty.any():
    display(pop.loc[same_ty, ["id", "title", "year", "fin_budget", "fin_revenue", "runtime", "in_tmdb"]].sort_values("title"))

### 3f. Release dates
TMDB's date drives the year split. Release month is an allowed input. Jan 1 is checked as a placeholder.

In [ ]:
d = pop["release_dt"]
jan1 = ((d.dt.month == 1) & (d.dt.day == 1)).sum()
print(f"release_dt: {d.min().date()} to {d.max().date()}, missing {d.isna().sum()}, Jan 1 dates {jan1} ({jan1 / len(d):.1%})")
if jan1 / len(d) > 0.01:
    flag("Unified", "release_dt", "placeholder date?", f"{jan1} films dated Jan 1")

month_share = pop["release_month"].value_counts(normalize=True).sort_index() * 100
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 3.2), width_ratios=[2, 1], layout="constrained")
for p, g in pop.groupby("period", observed=True):
    yc = g["year"].value_counts().sort_index()
    a1.bar(yc.index, yc.values, color=SERIES[p], width=0.8, label=f"{p} ({len(g):,})")
a1.set_title("Usable films per release year"); a1.set_xlabel("release year"); a1.set_ylabel("films")
a1.legend(fontsize=7, frameon=False); a1.grid(axis="x", visible=False)
a2.bar(month_share.index, month_share.values, color=BLUE, width=0.7)
a2.axhline(100 / 12, color=INK2, ls=":", lw=1)
a2.set_xticks(range(1, 13), list("JFMAMJJASOND"))
a2.set_title("Release month (dotted line = even 8.3%)"); a2.set_ylabel("% of films"); a2.grid(axis="x", visible=False)
plt.show()

### 3g. Do the sources agree? (new)

Where two sources report the same thing, they should agree. Disagreement is either a wrong match or a data error, and it tells Phase 3 which source to trust. Agreement = within 15% for money, 5 minutes for runtime, 7 days for release dates, same label for MPAA.

In [ ]:
def agree_num(a, b, tol):
    m = pop[a].gt(0) & pop[b].gt(0)
    r = (pop.loc[m, a] / pop.loc[m, b] - 1).abs()
    return {"films with both": int(m.sum()), "% agree": round(100 * (r <= tol).mean(), 1),
            "median |diff| %": round(100 * r.median(), 1)}

def agree_abs(a, b, tol):
    x, y = pop[a], pop[b]
    m = x.gt(0) & y.gt(0) if pd.api.types.is_numeric_dtype(x) else x.notna() & y.notna()
    diff = (x[m] - y[m]).abs()
    if hasattr(diff, "dt"):
        diff = diff.dt.days
    return {"films with both": int(m.sum()), "% agree": round(100 * (diff <= tol).mean(), 1),
            "median |diff| %": None}

agreement = pd.DataFrame({
    "budget: TMDB vs The Numbers": agree_num("budget", "tn_Production Budget (USD)", 0.15),
    "budget: TMDB vs Wikidata": agree_num("budget", "wd_wd_budget_usd", 0.15),
    "budget: The Numbers vs Wikidata": agree_num("tn_Production Budget (USD)", "wd_wd_budget_usd", 0.15),
    "revenue: TMDB vs The Numbers worldwide": agree_num("revenue", "tn_Worldwide Gross (USD)", 0.15),
    "revenue: TMDB vs Wikidata box office": agree_num("revenue", "wd_wd_box_office_usd", 0.15),
    "domestic: The Numbers vs Box Office Mojo": agree_num("tn_Domestic Gross (USD)", "bom_domestic_gross", 0.15),
    "domestic: The Numbers vs RT box office": agree_num("tn_Domestic Gross (USD)", "rt_boxOffice_usd", 0.15),
    "runtime: TMDB vs The Numbers (min)": agree_abs("runtime", "tn_Running Time (minutes)", 5),
    "runtime: TMDB vs RT (min)": agree_abs("runtime", "rt_runtimeMinutes", 5),
    "release: TMDB vs The Numbers (days)": agree_abs("release_dt", "tn_Release Date", 7),
    "release: TMDB vs Box Office Mojo first day (days)": agree_abs("release_dt", "bom_first_date", 7),
}).T
display(agreement)
for k, r in agreement.iterrows():
    if r["% agree"] < 80:
        flag("Unified", k, "sources disagree", f"{r['% agree']}% agree across {r['films with both']:,} films")

mpaa = pop[["tn_MPAA Rating", "rt_rating", "sl_content_rating"]].dropna()
print(f"MPAA rating, The Numbers vs RT vs RT sl (films with all three: {len(mpaa):,}): "
      f"all three agree on {(mpaa.nunique(axis=1) == 1).mean():.0%}")

# Matt's label risk: TMDB "worldwide" revenue that equals RT domestic box office
dom = pop["rt_boxOffice_usd"].gt(0) & pop["revenue"].gt(0)
same_dom = dom & (pop["revenue"] / pop["rt_boxOffice_usd"]).between(0.98, 1.02)
fixable = same_dom & pop["tn_Worldwide Gross (USD)"].gt(pop["revenue"] * 1.02)
print(f"TMDB revenue within 2% of RT domestic: {same_dom.sum():,} films (Matt: 396). "
      f"The Numbers has a higher worldwide gross for {fixable.sum():,} of them.")
flag("Unified", "fin_revenue", "domestic-only revenue?",
     f"{same_dom.sum():,} films have TMDB revenue = RT domestic; The Numbers worldwide is higher for {fixable.sum():,}")

ratio = (pop["budget"] / pop["tn_Production Budget (USD)"]).where(pop["budget"].gt(0) & pop["tn_Production Budget (USD)"].gt(0))
fig, ax = plt.subplots(figsize=(6, 3.2), layout="constrained")
ax.hist(np.log10(ratio.dropna()).clip(-1.5, 1.5), bins=60, color=BLUE, edgecolor="white", linewidth=0.4)
ax.axvspan(np.log10(1 / 1.15), np.log10(1.15), color=BLUE_LIGHT, zorder=0)
ax.set_title(f"Budget, TMDB / The Numbers (n = {ratio.notna().sum():,}; shaded = within 15%)")
ax.set_xlabel("log10 ratio (0 = identical)"); ax.set_ylabel("films")
plt.show()

### 3h. Who has which data? (new)
Matt found the RT join covers big films better than small ones (58% of $1–5M vs 91% of $60M+). The same check for every source and key input, by period and budget tier. Uneven coverage means missing values aren't random, so Phase 3 needs "unknown" indicators, not dropped rows.

In [ ]:
pop["budget_tier"] = pd.cut(pop["fin_budget"], [1e6, 5e6, 20e6, 60e6, np.inf], right=False,
                           labels=["$1-5M", "$5-20M", "$20-60M", "$60M+"])
cov_cols = {"TMDB genres": "genres", "The Numbers match": "tn_Movie Name", "Wikidata": "wd_item",
            "Box Office Mojo": "bom_title", "RT movies (director)": "rt_director", "RT writer": "rt_writer",
            "RT sl (actors)": "sl_actors", "MPAA (The Numbers)": "tn_MPAA Rating", "Source (The Numbers)": "tn_Source",
            "Franchise (The Numbers)": "tn_Franchise", "Critic reviews": "rtrev_n_reviews"}
cov = pd.concat({
    "period": pop.groupby("period", observed=True).agg(**{k: (v, lambda s: s.notna().mean()) for k, v in cov_cols.items()}),
    "budget tier": pop.groupby("budget_tier", observed=True).agg(**{k: (v, lambda s: s.notna().mean()) for k, v in cov_cols.items()}),
})
display((100 * cov).round(0).astype(int).astype(str) + "%")
gap = (cov.loc["budget tier"].max() - cov.loc["budget tier"].min()) * 100
for k, g in gap.items():
    if g > 20:
        flag("Unified", cov_cols[k], "coverage skews by budget", f"{k}: coverage gap of {g:.0f} points between budget tiers")

### 3i. Flag register
Every flag from 2d–3h plus the skew, rare-level and missingness flags from the profile tables. Hand-off to the readiness gate (step 12) and the cleaning ledger (step 13).

In [ ]:
reg = list(FLAGS)
for f_, r in prof_num.iterrows():
    if r["role"] in (A, CAND, O, "derived") and r["skew flag"]:
        reg.append({"file": "Unified", "field": f_, "issue": "skew > 0.5",
                    "evidence": f"skew {r['skew']:.2f}; median {r['median']:,.3g} vs mean {r['mean']:,.3g}"})
    if r["role"] in (A, CAND) and r["% missing"] > 0:
        reg.append({"file": "Unified", "field": f_, "issue": "missing", "evidence": f"{r['% missing']}% missing"})
for f_, r in prof_cat.iterrows():
    if r["role"] not in (A, CAND):
        continue
    if r["levels < 5% of rows"] > 0:
        reg.append({"file": "Unified", "field": f_, "issue": "rare levels",
                    "evidence": f"{r['levels < 5% of rows']:,} of {r['levels']:,} levels under 5% of rows"})
    if r["% missing"] > 0:
        reg.append({"file": "Unified", "field": f_, "issue": "missing", "evidence": f"{r['% missing']}% missing"})
register = pd.DataFrame(reg).drop_duplicates(["file", "field", "issue"]).sort_values(["issue", "field"])
with pd.option_context("display.max_rows", 150, "display.max_colwidth", 100):
    display(register.reset_index(drop=True))
print(f"{len(register)} flags across {register['field'].nunique()} fields")
register.groupby("issue").size().rename("flags").to_frame()

### 3j. What the profile shows (facts from the outputs above, run 2026-11-23)

**Population and the label**
1. **6,838 usable films** (train 4,952 · test 1,264 · stress 622), up from Matt's 5,766. The test set grows 16% (1,086 → 1,264), which tightens the bootstrap CI on the evidence standard.
2. **The added films are mostly losers.** Films whose budget or revenue came from The Numbers or Wikidata are 83% losers at 2.5×, against 65% for Matt's TMDB-only films. Overall: 67.5% (Matt 64.6%). This is the S4 worry in reverse: the extra sources recover some of the flops TMDB was missing. The 20%-capacity ceiling drops slightly, from 31% to 30% of losers.
3. **S4 is still triggered.** 39.0% of films with a budget ≥ $1M still have no revenue from any source (Matt: 43.6%), and 52% in the stress years.
4. **Junk values grew with the population.** 269 films earn < 1% of budget (Matt 179), 35 report revenue < $1,000, 380 < $100,000. The lowest are the same junk Matt found (*Aquajots*: $100M budget, $1 revenue). Six films above 100×, including *Vacancy* (2018, $903M on $1M) and a new one, *Space Station 3D*, where a Wikidata budget meets a TMDB revenue.

**Do the sources agree?** (new)
5. **Money mostly agrees.** Budget: TMDB vs The Numbers 85%, TMDB vs Wikidata 90% (within 15%). Worldwide revenue: 81–87%. Domestic gross across The Numbers, Box Office Mojo and RT: 92–97%. Runtime: 92–93% within 5 minutes. MPAA rating: the three sources agree on 99%.
6. **Release dates often don't.** TMDB's date is within 7 days of The Numbers / Box Office Mojo for only 73–75% of films. TMDB often records the first release anywhere (a festival or a foreign market); the others record the US release. Release month is an allowed input, so Phase 3 has to pick one definition.
7. **Matt's domestic-as-worldwide risk is larger than first counted:** 434 films (Matt: 396) have TMDB revenue equal to RT domestic. **The Numbers has a higher worldwide figure for 220 of them**, a direct repair.

**Coverage is uneven** (new)
8. **Every extra source covers big films far better than small ones.** The Numbers matches 37% of $1–5M films vs 91% of $60M+; MPAA and source material 35% vs 91%; director 62% vs 90%; franchise 3% vs 44%. Small films are where most losers are, so a missing value is not random. RT stefanoleone992 (scraped in 2020) has actors for only 5% of stress-period films.
9. **Franchise is thin and partly garbled:** 15% coverage; 92 of 1,006 values look like several franchise names run together (e.g. "BatmanDark Knight Trilogy").
10. **Track records are sparse:** 63% of directors and 67% of production companies appear in only one usable film.

**Keys and duplicates**
11. **Duplicates (fixed in the build, 2026-11-23).** The first profile found title+year pairs that were the same film entered twice (*Spider-Man: No Way Home*, *Timber Falls*): the build's rule required runtime within 5 minutes, and one copy had runtime 0. The rule now treats 0 as unknown, which also caught *WALL·E* and *Black Panther: Wakanda Forever*. 3 pairs remain and are **not** duplicates: *Underdogs* (two different 2013 films), *Uncharted 2* (a video game, entered twice) and a junk *Blue Beetle* entry ($400M revenue, runtime 0). Those are cleaning issues. 2 usable films share a Wikidata item.
12. **Missed The Numbers matches (fixed in the build, 2026-11-23).** About 120 The Numbers films had been added as separate "not in TMDB" rows although TMDB had them under a differently written title (*Star Wars **Ep. VII**: The Force Awakens*, *Kill Bill: **Volume** 1*, *Harry Potter and the **Sorcerer's** Stone*). A second matching pass (build §4b) now joins them, which removed about 90 double-counted usable films (6,928 → 6,838). About 10 remain where the titles share almost nothing (*X-Men 2* vs *X2*, *F9: The Fast Saga* vs *F9*).
13. **Review copies (fixed in the build).** The stefanoleone992 review file had 120,945 rows identical in every column, and the andrezaza file 12,394 repeated review ids (each repeat identical in every column). Only those copies are dropped; every distinct review is kept. The individual reviews are now saved as their own table, `reviews_unified.csv` (one row per review, keyed by `film_uid`).

**Raw-file notes:** The Numbers has 150 release dates after the access date (up to 2068); none reach the usable population. RT sl reviews have 83 placeholder dates (1800-01-01). The Box Office Mojo scraper has no license.

### 3k. Proposed dispositions and phase gate (DRAFT, pending Tanner + Matt sign-off)

Matt's §3k rows 1–12 still apply. These are the additions for the unified dataset. **None are applied here.**

| # | Flag | Evidence | Proposed disposition | Treatment | Verification |
|---|---|---|---|---|---|
| U1 | Population now multi-source | 6,838 films; 1,078 use The Numbers / Wikidata financials | Requires a contract amendment | Dated amendment: population = 2000–2023, budget ≥ $1M, revenue > 0 **from TMDB, The Numbers or Wikidata**, with source precedence stated | Amendment recorded in the contract file |
| U2 | Added films skew to losers | 83% vs 65% losers | Usable; report it | Report results on the full population and on Matt's TMDB-only population as a sensitivity check | Capture on both populations |
| U3 | Domestic-as-worldwide revenue | 434 films; The Numbers higher for 220 | Requires repair | Replace TMDB revenue with The Numbers worldwide where TMDB = RT domestic and The Numbers is higher. Flag the other 214. | Loser rate before / after |
| U4 | Source disagreement on budget / revenue | 81–90% agree within 15% | Requires a rule | Keep precedence TMDB → The Numbers → Wikidata, but hand-check films where they disagree by > 2× | Hand-check sample error rate |
| U5 | Release date definitions differ | 74% agree within 7 days | Requires a rule | Use the US release date (The Numbers / Box Office Mojo) where available, TMDB otherwise; keep the source | Month distribution before / after |
| U6 | Coverage skews by budget | The Numbers 37% vs 91%; director 62% vs 90% | Usable with a limitation | Every optional input gets an "unknown" indicator; no dropped rows | Loser rate by known / unknown |
| U7 | Leftover duplicates | Runtime 0 blocked the merge rule | **Done** (build, 2026-11-23) | Runtime 0 counts as unknown; 4 more films merged | 3 same-title pairs left, none duplicates. Second pass (build §4b) joined ~120 missed The Numbers matches; ~10 remain (*X-Men 2* / *X2*) |
| U8 | Review copies | 120,945 + 12,394 copies of the same review | **Done** (build, 2026-11-23) | Only identical copies dropped; all reviews saved to `reviews_unified.csv` | Counts reported in the build |
| U11 | Junk entries | *Uncharted 2* (video game), *Blue Beetle* copy with $400M revenue and runtime 0 | Requires repair | Cleaning rule for runtime-0 TMDB entries that collide with a real film | Pair count after cleaning |
| U9 | Franchise field | 14% coverage, 92 garbled values | Investigate | See `FEATURE_IDEAS.md` (sequel / franchise); likely needs Wikidata series data | Coverage after |
| U10 | Candidate inputs not in the contract | MPAA, source, creative type, writer, cast, franchise | Needs a decision | Decide which to add by amendment before Phase 3 features | Decision recorded |

**Proposed phase gate: CONTINUE**, after U1 (amendment). Build fixes U7 and U8 are done. The S4 readiness verdict still belongs to step 12.